# NB09: Manuscript compilation (Phase 9R)

Builds all manuscript tables (1 through 11), the figure inventory, and the
key-results manifest by reading persisted CSV/JSON outputs from Phases 1-11.
All winning-feature-set references come from the Phase 3R winning
configuration JSON via src.plot_style.load_winning_config - nothing is
hard-coded to 'raw', 'alr_aug', 'pwlr_aug', etc.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
from src.plot_style import load_winning_config
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# v7 Part H: per-family canonical spec replaces the legacy
# legacy pre-v7 single-winner assumption. Forest / T_C / opx_liq is the
# primary single-feature-set stand-in for notebooks that have not
# been restructured into per-family output blocks.
from src.data import canonical_model_spec
_spec_primary = canonical_model_spec('T_C', 'opx_liq', 'forest', RESULTS)
WIN_FEAT = _spec_primary['feature_set']
print(f'v7 primary (forest / T_C / opx_liq) feature set: {WIN_FEAT}')
print(f'Phase 3R winning feature set: {WIN_FEAT}')

## Phase 9.EXT: Extended analyses (absorbed from former NB10)

**Purpose:** Run the downstream analyses that used to live in NB10 so NB09 is
self-contained - one compute-and-compile notebook rather than two. The
artifact CSVs (`nb10_*.csv`, `model_IsolationForest_opx_liq.joblib`,
`model_RF_*_opx_liq_H2O.joblib`) are still named with the `nb10_` prefix so
every existing manuscript reference keeps resolving.

Analyses: 10.1 two-pyroxene Thermobar baseline; 10.2 H2O residual dependence;
10.2b H2O-engineered retrain; 10.3 IQR uncertainty; 10.4 analytical-noise MC;
10.5 IsolationForest OOD filter; 10.6 OOD-paradox method comparison.

In [ ]:
from src.data import canonical_model_spec
# Phase 9.EXT setup: extra imports + canonical model/data loads for the
# absorbed NB10 analyses. NB09's earlier imports supply ROOT, DATA_PROC,
# DATA_SPLITS, MODELS, FIGURES, RESULTS, WIN_FEAT, numpy as np, pandas as pd.

from src.features import build_feature_matrix
from src.models import predict_median, predict_iqr
from src.evaluation import compute_metrics as metrics
from src.plot_style import canonical_model_filename, apply_style
apply_style()

import joblib
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

# v7 per-family canonical spec: T and P forests may use different feature
# sets (T_C opx_liq = alr / P_kbar opx_liq = raw). Build one feat_fn per
# target and wire X_*_T vs X_*_P everywhere so both model.predict() calls
# get the matrix their trained shape expects.
_spec_T_cell3 = canonical_model_spec('T_C',    'opx_liq', 'forest', RESULTS)
_spec_P_cell3 = canonical_model_spec('P_kbar', 'opx_liq', 'forest', RESULTS)
WIN_FEAT_T = _spec_T_cell3['feature_set']
WIN_FEAT_P = _spec_P_cell3['feature_set']
feat_fn_T = lambda df, use_liq: build_feature_matrix(df, WIN_FEAT_T, use_liq=use_liq)
feat_fn_P = lambda df, use_liq: build_feature_matrix(df, WIN_FEAT_P, use_liq=use_liq)
feat_fn = feat_fn_T  # back-compat: legacy calls default to T winner

model_T = joblib.load(MODELS / canonical_model_filename('T_C', 'opx_liq', 'forest', RESULTS))
model_P = joblib.load(MODELS / canonical_model_filename('P_kbar', 'opx_liq', 'forest', RESULTS))
qrf_T_path = MODELS / 'model_QRF_T_C_opx_liq.joblib'
qrf_P_path = MODELS / 'model_QRF_P_kbar_opx_liq.joblib'
qrf_T = joblib.load(qrf_T_path) if qrf_T_path.exists() else None
qrf_P = joblib.load(qrf_P_path) if qrf_P_path.exists() else None

df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
idx_tr = np.load(DATA_SPLITS / 'train_indices_opx_liq.npy')
idx_te = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
df_train = df_liq.loc[idx_tr].copy()
df_test  = df_liq.loc[idx_te].copy()
X_train_T, feat_names_T = feat_fn_T(df_train, use_liq=True)
X_train_P, feat_names_P = feat_fn_P(df_train, use_liq=True)
X_test_T,  _            = feat_fn_T(df_test,  use_liq=True)
X_test_P,  _            = feat_fn_P(df_test,  use_liq=True)
# Legacy aliases for target-agnostic code paths (IsoForest fit, PCA).
X_train, feat_names = X_train_T, feat_names_T
X_test              = X_test_T

arcpl_path = RESULTS / 'nb04b_arcpl_predictions.csv'
if not arcpl_path.exists():
    raise FileNotFoundError(f'Missing {arcpl_path}. Run nb04b first.')
df_arcpl = pd.read_csv(arcpl_path)
# ArcPL comes from nb04b with the training-schema columns already applied,
# so build_feature_matrix can be called directly.
X_arcpl_T, _ = feat_fn_T(df_arcpl, use_liq=True)
X_arcpl_P, _ = feat_fn_P(df_arcpl, use_liq=True)
X_arcpl      = X_arcpl_T  # legacy alias
print(f'Ext setup OK: train n={len(df_train)} test n={len(df_test)} ArcPL n={len(df_arcpl)}')
print(f'  feature sets: T={WIN_FEAT_T} ({X_train_T.shape[1]})  P={WIN_FEAT_P} ({X_train_P.shape[1]})')


In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.1: Two-pyroxene benchmark via Thermobar on the opx-liq test set
# Evaluates Putirka (2008) Eq. 36/39 two-pyroxene-only T (and P where available)
# as a non-ML baseline that can be directly compared against our opx-liq RF.

try:
    import Thermobar as pt
    HAVE_THERMOBAR = True
except Exception as e:
    HAVE_THERMOBAR = False
    print(f'Thermobar unavailable: {e}')

two_px_rows = []

if HAVE_THERMOBAR and {'T_C', 'P_kbar'}.issubset(df_test.columns):
    # Build Thermobar-style input frames. The two-pyroxene thermometer needs
    # both Opx and Cpx compositions; samples without Cpx are skipped.
    opx_cols = ['SiO2', 'TiO2', 'Al2O3', 'FeO_total', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'Cr2O3']
    cpx_prefix = 'cpx_'
    has_cpx = any(c.startswith(cpx_prefix) for c in df_test.columns)
    if has_cpx:
        mask = df_test[[f'{cpx_prefix}SiO2']].notna().all(axis=1).values
        sub = df_test[mask].copy()
        opx_df = pd.DataFrame({f'{c}_Opx': sub[c].values for c in opx_cols if c in sub.columns})
        cpx_df = pd.DataFrame({f'{c}_Cpx': sub[f'{cpx_prefix}{c}'].values
                               for c in opx_cols if f'{cpx_prefix}{c}' in sub.columns})

        try:
            out = pt.calculate_cpx_opx_press_temp(
                cpx_comps=cpx_df, opx_comps=opx_df,
                equationT='T_Put2008_eq36', equationP='P_Put2008_eq39')
            t_pred = out['T_K_calc'].values - 273.15
            p_pred = out['P_kbar_calc'].values if 'P_kbar_calc' in out.columns else np.full(len(sub), np.nan)
            t_obs = sub['T_C'].values
            p_obs = sub['P_kbar'].values

            two_px_rows.append({'target': 'T_C', **metrics(t_obs[np.isfinite(t_pred)],
                                                           t_pred[np.isfinite(t_pred)])})
            mask_p = np.isfinite(p_pred)
            if mask_p.sum() > 0:
                two_px_rows.append({'target': 'P_kbar', **metrics(p_obs[mask_p], p_pred[mask_p])})
        except Exception as e:
            print(f'Thermobar call failed: {e}')
    else:
        print('Test set has no Cpx columns; two-pyroxene benchmark skipped.')
else:
    print('Skipped two-pyroxene benchmark (no Thermobar or missing cols).')

two_px_df = pd.DataFrame(two_px_rows)
two_px_df.to_csv(RESULTS / 'nb10_two_pyroxene_benchmark.csv', index=False)
if len(two_px_df):
    print(two_px_df.round(3).to_string(index=False))

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.2: H2O dependence in ArcPL residuals
# Use canonical RF T and P on ArcPL, compute residuals, and regress against
# liquid H2O to see whether our dry-trained model has a systematic wet bias.

from scipy.stats import pearsonr, spearmanr

h2o_rows = []
if 'H2O_Liq' in df_arcpl.columns or 'liq_H2O' in df_arcpl.columns:
    h2o_col = 'liq_H2O' if 'liq_H2O' in df_arcpl.columns else 'H2O_Liq'
    # Per-target feature matrices (T=alr/P=raw for opx_liq forest).
    t_pred = predict_median(model_T, X_arcpl_T)
    p_pred = predict_median(model_P, X_arcpl_P)

    h2o = df_arcpl[h2o_col].values.astype(float)

    for tgt, obs_col, pred in [('T_C', 'T_C', t_pred), ('P_kbar', 'P_kbar', p_pred)]:
        if obs_col not in df_arcpl.columns:
            continue
        obs = df_arcpl[obs_col].values.astype(float)
        resid = pred - obs
        mask = np.isfinite(resid) & np.isfinite(h2o)
        if mask.sum() < 5:
            continue
        r_p, p_p = pearsonr(h2o[mask], resid[mask])
        r_s, p_s = spearmanr(h2o[mask], resid[mask])
        slope, intercept = np.polyfit(h2o[mask], resid[mask], 1)
        h2o_rows.append({
            'target':      tgt,
            'n':           int(mask.sum()),
            'pearson_r':   float(r_p),
            'pearson_p':   float(p_p),
            'spearman_r':  float(r_s),
            'spearman_p':  float(p_s),
            'slope':       float(slope),
            'intercept':   float(intercept),
            'mean_resid_dry': float(np.mean(resid[mask][h2o[mask] < 1.0])) if (h2o[mask] < 1.0).any() else np.nan,
            'mean_resid_wet': float(np.mean(resid[mask][h2o[mask] >= 1.0])) if (h2o[mask] >= 1.0).any() else np.nan,
        })
else:
    print('ArcPL has no H2O_liq or liq_H2O column; H2O analysis skipped.')

h2o_df = pd.DataFrame(h2o_rows)
h2o_df.to_csv(RESULTS / 'nb10_h2o_dependence.csv', index=False)
if len(h2o_df):
    print(h2o_df.round(4).to_string(index=False))

## Phase 9.2b: H2O-aware engineered features (v5)

Phase 10.2 above shows the canonical dry-trained RF has a residual-H2O
correlation. Part 6 of the v5 plan adds four engineered features that encode
water explicitly and retrains the opx-liq RF:

- `H2O_over_SiO2 = H2O / liq_SiO2`  (bulk melt water mass ratio)
- `H2O_activity_proxy = H2O / (H2O + liq_SiO2)`  (mole-fraction-ish activity)
- `log1p_H2O = log(1 + H2O)`  (compresses high-water tails)
- `H2O_x_Mg_num_liq = H2O * Mg_num_liq`  (water moderated by liquid Fe/Mg)

Samples with missing `H2O_Liq` are treated as dry (0 wt%) and extreme values
above 15 wt% (30 / 166 rows - almost certainly unit errors, likely ppm vs
wt%) are clipped to 15. The test: if residual-H2O Pearson |r| on ArcPL drops
below 0.1 after retraining, the engineered features successfully absorbed
the water signal and the residual correlation in the canonical RF is a
feature-representation gap, not an intrinsic thermobarometric limit.

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.2b (v5): H2O engineered features + opx-liq RF retrain

def _clean_h2o(vals):
    """Treat NaN as dry (0 wt%), clip plausible range to [0, 15] wt%."""
    v = np.where(np.isfinite(vals), vals, 0.0).astype(float)
    return np.clip(v, 0.0, 15.0)


def _add_h2o_engineered(df, X, feat_names):
    """Append 4 H2O-aware features to an existing X matrix.
    Missing liq cols default to neutral values so the feature never throws."""
    h2o = _clean_h2o(df['H2O_Liq'].values if 'H2O_Liq' in df.columns else np.zeros(len(df)))
    sio2 = df['liq_SiO2'].fillna(50.0).values if 'liq_SiO2' in df.columns else np.full(len(df), 50.0)
    mgo  = df['liq_MgO'].fillna(0.0).values  if 'liq_MgO'  in df.columns else np.zeros(len(df))
    feo  = df['liq_FeO'].fillna(0.0).values  if 'liq_FeO'  in df.columns else np.zeros(len(df))
    mg_num_liq = np.where((mgo + feo) > 1e-6, mgo / (mgo + feo), 0.5)

    new_cols = np.column_stack([
        h2o / np.maximum(sio2, 1e-6),
        h2o / np.maximum(h2o + sio2, 1e-6),
        np.log1p(h2o),
        h2o * mg_num_liq,
    ])
    new_names = ['H2O_over_SiO2', 'H2O_activity_proxy', 'log1p_H2O', 'H2O_x_Mg_num_liq']
    X_aug = np.column_stack([X, new_cols]).astype(float)
    # Guard against any NaN/Inf that survive (conservative: replace with 0)
    X_aug = np.where(np.isfinite(X_aug), X_aug, 0.0)
    return X_aug, list(feat_names) + new_names


X_train_h2o_T, feat_h2o_T = _add_h2o_engineered(df_train, X_train_T, feat_names_T)
X_test_h2o_T,  _          = _add_h2o_engineered(df_test,  X_test_T,  feat_names_T)
X_train_h2o_P, feat_h2o_P = _add_h2o_engineered(df_train, X_train_P, feat_names_P)
X_test_h2o_P,  _          = _add_h2o_engineered(df_test,  X_test_P,  feat_names_P)
# Legacy aliases (T is the canonical primary for feat-count reporting).
X_train_h2o, feat_h2o = X_train_h2o_T, feat_h2o_T
X_test_h2o            = X_test_h2o_T
print(f'Augmented feature count: T {X_train_T.shape[1]} -> {X_train_h2o_T.shape[1]} | '
      f'P {X_train_P.shape[1]} -> {X_train_h2o_P.shape[1]}')

# Retrain opx-liq RF with the same hyperparameters as the canonical model.
# We reuse canonical model params (n_estimators, min_samples_leaf, max_features)
# so any change in test RMSE is attributable to the added features, not tuning.
def _extract_params(fitted_rf):
    p = fitted_rf.get_params()
    return {k: p[k] for k in ('n_estimators', 'min_samples_leaf', 'max_features', 'max_depth')
            if k in p}

from sklearn.ensemble import RandomForestRegressor

rf_T_h2o = RandomForestRegressor(**_extract_params(model_T),
                                 random_state=SEED_MODEL, n_jobs=-1)
rf_P_h2o = RandomForestRegressor(**_extract_params(model_P),
                                 random_state=SEED_MODEL, n_jobs=-1)
rf_T_h2o.fit(X_train_h2o_T, df_train['T_C'].values)
rf_P_h2o.fit(X_train_h2o_P, df_train['P_kbar'].values)

# Evaluate on ArcPL (same residual-H2O correlation check as Phase 10.2).
h2o_rows2 = []
if 'H2O_Liq' in df_arcpl.columns or 'liq_H2O' in df_arcpl.columns:
    h2o_col = 'liq_H2O' if 'liq_H2O' in df_arcpl.columns else 'H2O_Liq'
    X_arcpl_h2o_T, _ = _add_h2o_engineered(df_arcpl, X_arcpl_T, feat_names_T)
    X_arcpl_h2o_P, _ = _add_h2o_engineered(df_arcpl, X_arcpl_P, feat_names_P)
    t_pred_h2o = predict_median(rf_T_h2o, X_arcpl_h2o_T)
    p_pred_h2o = predict_median(rf_P_h2o, X_arcpl_h2o_P)
    h2o_vals = _clean_h2o(df_arcpl[h2o_col].values.astype(float))

    for tgt, obs_col, pred in [('T_C', 'T_C', t_pred_h2o), ('P_kbar', 'P_kbar', p_pred_h2o)]:
        if obs_col not in df_arcpl.columns:
            continue
        obs = df_arcpl[obs_col].values.astype(float)
        resid = pred - obs
        mask = np.isfinite(resid) & np.isfinite(h2o_vals)
        if mask.sum() < 5:
            continue
        r_p, p_p = pearsonr(h2o_vals[mask], resid[mask])
        h2o_rows2.append({
            'target': tgt, 'n': int(mask.sum()),
            'pearson_r_after':  float(r_p),
            'pearson_p_after':  float(p_p),
            'rmse_after':       float(np.sqrt(mean_squared_error(obs[mask], pred[mask]))),
            'passes_threshold': bool(abs(r_p) < 0.10),
        })

# Evaluate on held-out test set: does adding H2O features change test RMSE?
rmse_rows = []
for tgt, y_te, pred_base, pred_h2o in [
    ('T_C',    df_test['T_C'].values, predict_median(model_T, X_test_T),
     predict_median(rf_T_h2o, X_test_h2o_T)),
    ('P_kbar', df_test['P_kbar'].values, predict_median(model_P, X_test_P),
     predict_median(rf_P_h2o, X_test_h2o_P)),
]:
    rmse_rows.append({
        'target': tgt,
        'rmse_baseline': float(np.sqrt(mean_squared_error(y_te, pred_base))),
        'rmse_with_h2o': float(np.sqrt(mean_squared_error(y_te, pred_h2o))),
        'r2_baseline':   float(r2_score(y_te, pred_base)),
        'r2_with_h2o':   float(r2_score(y_te, pred_h2o)),
    })

h2o2_df = pd.DataFrame(h2o_rows2)
rmse_df = pd.DataFrame(rmse_rows)
h2o2_df.to_csv(RESULTS / 'nb10_h2o_engineered_arcpl.csv', index=False)
rmse_df.to_csv(RESULTS / 'nb10_h2o_engineered_test_rmse.csv', index=False)

# Persist the retrained models so NB08 / NBF can pick them up if needed.
joblib.dump(rf_T_h2o, MODELS / 'model_RF_T_C_opx_liq_H2O.joblib')
joblib.dump(rf_P_h2o, MODELS / 'model_RF_P_kbar_opx_liq_H2O.joblib')

print('Residual-H2O correlation after retrain with H2O features:')
if len(h2o2_df):
    print(h2o2_df.round(4).to_string(index=False))
print('\nTest-set RMSE (baseline vs with H2O):')
print(rmse_df.round(3).to_string(index=False))
print('\nBefore (Phase 10.2):')
if len(h2o_df):
    print(h2o_df[['target', 'n', 'pearson_r', 'pearson_p']].round(4).to_string(index=False))

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.3: IQR per-sample uncertainty
# Uses predict_iqr (16th to 84th percentile across trees) as a cheap
# epistemic uncertainty surrogate and correlates it with |residual| on the
# held-out test set.

_, q16_T, q84_T = predict_iqr(model_T, X_test_T)
_, q16_P, q84_P = predict_iqr(model_P, X_test_P)
test_pred_T = predict_median(model_T, X_test_T)
test_pred_P = predict_median(model_P, X_test_P)
iqr_T = q84_T - q16_T
iqr_P = q84_P - q16_P

iqr_rows = []
for tgt, y_true, pred, iqr_vals in [
    ('T_C',    df_test['T_C'].values, test_pred_T, iqr_T),
    ('P_kbar', df_test['P_kbar'].values, test_pred_P, iqr_P),
]:
    abs_res = np.abs(pred - y_true)
    r_p, _ = pearsonr(iqr_vals, abs_res)
    r_s, _ = spearmanr(iqr_vals, abs_res)
    # Calibration: fraction of samples where |residual| <= IQR/2
    calib = float(np.mean(abs_res <= iqr_vals / 2.0))
    iqr_rows.append({
        'target':        tgt,
        'iqr_mean':      float(iqr_vals.mean()),
        'iqr_median':    float(np.median(iqr_vals)),
        'pearson_iqr_vs_absres':  float(r_p),
        'spearman_iqr_vs_absres': float(r_s),
        'frac_abs_res_le_halfiqr': calib,
        'n': int(len(y_true)),
    })

iqr_df = pd.DataFrame(iqr_rows)
iqr_df.to_csv(RESULTS / 'nb10_iqr_uncertainty.csv', index=False)
print(iqr_df.round(3).to_string(index=False))


In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
from tqdm.auto import tqdm
# Phase 10.4: Analytical-noise propagation (EPMA measurement uncertainty)
# Perturb each test sample's oxide columns with the analytical-noise model
# used in Phase 3R (3% rel on majors, 8% on minors) many times, push each
# perturbed replicate through the canonical RF, and report the standard
# deviation of the resulting prediction distribution. This yields a
# propagated analytical uncertainty that is independent of the RF's internal
# tree variance.

N_MC = 200
MAJ_REL = 0.03
MIN_REL = 0.08
MAJORS = {'SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO', 'liq_SiO2', 'liq_Al2O3',
          'liq_FeO', 'liq_MgO', 'liq_CaO'}

rng = np.random.default_rng(SEED_NOISE_AUG)

mc_T = np.zeros((len(df_test), N_MC))
mc_P = np.zeros((len(df_test), N_MC))

all_ox_cols = [c for c in df_test.columns
               if c in OPX_FULL_OXIDES or c in {f'liq_{o}' for o in LIQ_OXIDES}]

for k in tqdm(range(N_MC), desc='analytical-noise propagation'):
    df_perturb = df_test.copy()
    for col in all_ox_cols:
        rel = MAJ_REL if col in MAJORS else MIN_REL
        noise = rng.normal(loc=0.0, scale=rel, size=len(df_perturb))
        df_perturb[col] = df_perturb[col].values * (1.0 + noise)
    X_mc_T, _ = feat_fn_T(df_perturb, use_liq=True)
    X_mc_P, _ = feat_fn_P(df_perturb, use_liq=True)
    mc_T[:, k] = predict_median(model_T, X_mc_T)
    mc_P[:, k] = predict_median(model_P, X_mc_P)

sigma_T = mc_T.std(axis=1)
sigma_P = mc_P.std(axis=1)

mc_rows = [
    {'target': 'T_C',
     'sigma_mean': float(sigma_T.mean()),
     'sigma_median': float(np.median(sigma_T)),
     'sigma_p90': float(np.percentile(sigma_T, 90)),
     'n_mc': N_MC,
     'n': int(len(df_test))},
    {'target': 'P_kbar',
     'sigma_mean': float(sigma_P.mean()),
     'sigma_median': float(np.median(sigma_P)),
     'sigma_p90': float(np.percentile(sigma_P, 90)),
     'n_mc': N_MC,
     'n': int(len(df_test))},
]
mc_df = pd.DataFrame(mc_rows)
mc_df.to_csv(RESULTS / 'nb10_analytical_uncertainty.csv', index=False)

# Per-sample MC predictions for optional downstream figures
mc_pred = pd.DataFrame({
    'index':    idx_te,
    'T_mean':   mc_T.mean(axis=1),
    'T_sigma':  sigma_T,
    'P_mean':   mc_P.mean(axis=1),
    'P_sigma':  sigma_P,
})
mc_pred.to_csv(RESULTS / 'nb10_mc_per_sample.csv', index=False)
print(mc_df.round(3).to_string(index=False))

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.5: IsolationForest out-of-distribution filter on ArcPL
# Fit IsolationForest on training-set features, score ArcPL samples, and
# recompute T and P RMSE on the in-distribution subset.

# v8-fix: contamination='auto' often labels 0 or ~50% outliers on this
# composition space. Score the training set to pick a data-driven cutoff
# (bottom 10% of TRAINING scores) so the ArcPL OOD bucket is a sensible
# minority (~5-25%) rather than 0% or 100%.
iso = IsolationForest(n_estimators=300, contamination='auto',
                      random_state=SEED_MODEL, n_jobs=-1)
iso.fit(X_train)
_train_score = iso.score_samples(X_train)
_ood_threshold = float(np.percentile(_train_score, 10))
arcpl_score = iso.score_samples(X_arcpl_T)
arcpl_label = np.where(arcpl_score < _ood_threshold, -1, 1)
print(f'v8-check OOD: training 10th percentile cutoff={_ood_threshold:.3f} | '
      f'ArcPL score min={arcpl_score.min():.3f} max={arcpl_score.max():.3f} | '
      f'OOD n={int((arcpl_label == -1).sum())}/{len(arcpl_label)} '
      f'({100*(arcpl_label == -1).mean():.1f}%)')

pred_T_arcpl = predict_median(model_T, X_arcpl_T)
pred_P_arcpl = predict_median(model_P, X_arcpl_P)

ood_rows = []
for tgt, obs_col, pred in [('T_C', 'T_C', pred_T_arcpl),
                            ('P_kbar', 'P_kbar', pred_P_arcpl)]:
    if obs_col not in df_arcpl.columns:
        continue
    obs = df_arcpl[obs_col].values.astype(float)
    mask_all = np.isfinite(obs)
    mask_in = mask_all & (arcpl_label == 1)
    mask_out = mask_all & (arcpl_label == -1)
    ood_rows.append({'target': tgt, 'subset': 'all',
                     **metrics(obs[mask_all], pred[mask_all])})
    if mask_in.sum() > 0:
        ood_rows.append({'target': tgt, 'subset': 'in_distribution',
                         **metrics(obs[mask_in], pred[mask_in])})
    if mask_out.sum() > 0:
        ood_rows.append({'target': tgt, 'subset': 'out_of_distribution',
                         **metrics(obs[mask_out], pred[mask_out])})

ood_df = pd.DataFrame(ood_rows)
ood_df.to_csv(RESULTS / 'nb10_ood_isoforest.csv', index=False)

# Persist per-sample OOD scores for downstream figures
arcpl_scores_df = pd.DataFrame({
    'index':        np.arange(len(df_arcpl)),
    'iso_score':    arcpl_score,
    'iso_inlier':   (arcpl_label == 1).astype(int),
    'T_pred':       pred_T_arcpl,
    'P_pred':       pred_P_arcpl,
})
arcpl_scores_df.to_csv(RESULTS / 'nb10_arcpl_ood_scores.csv', index=False)
joblib.dump(iso, MODELS / 'model_IsolationForest_opx_liq.joblib')

print(ood_df.round(3).to_string(index=False))

## Phase 9.6: OOD paradox diagnosis (v5)

The canonical dry-trained RF shows a paradox on ArcPL: IsolationForest labels
a sizable subset OOD, but the OOD subset does **not** consistently have
larger absolute residuals than the in-distribution subset. If an OOD flag
does not track error it is a diagnostic without predictive value.

Part 7 compares three OOD scoring methods on ArcPL by correlating each
anomaly score with |residual| on T and P (Spearman + Pearson):

1. **IsolationForest** score on the 1664-dim canonical feature space
2. **Mahalanobis distance** in the first 5 principal components of the
   training feature space (linear manifold distance)
3. **QRF sigma** = 0.5 * (q84 - q16) from the canonical QRF as a direct
   epistemic-uncertainty surrogate

The method with the largest |Spearman r| vs |residual| is the one that
actually predicts prediction error and is therefore canonized for NB08
downstream filtering. This is an **OPERATOR DECISION** point (A / B / C).

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 10.6 (v5): compare OOD scoring methods by |residual| correlation on ArcPL.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import mahalanobis

# 1) IsolationForest anomaly: negate score_samples so larger = more anomalous
iso_anomaly_arcpl = -arcpl_score

# 2) Mahalanobis in PC5 space of training features
_scaler = StandardScaler().fit(X_train)
X_tr_std = _scaler.transform(X_train)
X_ar_std = _scaler.transform(X_arcpl)
_pca = PCA(n_components=min(5, X_train.shape[1])).fit(X_tr_std)
Z_tr = _pca.transform(X_tr_std)
Z_ar = _pca.transform(X_ar_std)
mu = Z_tr.mean(axis=0)
cov = np.cov(Z_tr, rowvar=False)
cov_inv = np.linalg.pinv(cov + 1e-8 * np.eye(cov.shape[0]))
mahal = np.array([float(mahalanobis(z, mu, cov_inv)) for z in Z_ar])

# 3) QRF sigma (half IQR) on ArcPL - epistemic uncertainty surrogate
qrf_sigma_T = np.full(len(df_arcpl), np.nan)
qrf_sigma_P = np.full(len(df_arcpl), np.nan)
if qrf_T is not None and qrf_P is not None:
    q_T_ar = qrf_T.predict(X_arcpl_T, quantiles=[0.16, 0.5, 0.84])
    q_P_ar = qrf_P.predict(X_arcpl_P, quantiles=[0.16, 0.5, 0.84])
    qrf_sigma_T = 0.5 * (q_T_ar[:, 2] - q_T_ar[:, 0])
    qrf_sigma_P = 0.5 * (q_P_ar[:, 2] - q_P_ar[:, 0])

# Observed residuals on ArcPL (using canonical RF baseline, not H2O-retrain)
resid_T_arcpl = pred_T_arcpl - df_arcpl['T_C'].values.astype(float)
resid_P_arcpl = pred_P_arcpl - df_arcpl['P_kbar'].values.astype(float)

def _corr(score, absres):
    mask = np.isfinite(score) & np.isfinite(absres)
    if mask.sum() < 10:
        return {'spearman_r': np.nan, 'spearman_p': np.nan,
                'pearson_r':  np.nan, 'pearson_p':  np.nan, 'n': int(mask.sum())}
    s_r, s_p = spearmanr(score[mask], absres[mask])
    p_r, p_p = pearsonr (score[mask], absres[mask])
    return {'spearman_r': float(s_r), 'spearman_p': float(s_p),
            'pearson_r':  float(p_r), 'pearson_p':  float(p_p),
            'n': int(mask.sum())}


rows = []
for method, score_T, score_P in [
    ('isoforest',   iso_anomaly_arcpl, iso_anomaly_arcpl),
    ('mahalanobis', mahal,             mahal),
    ('qrf_sigma',   qrf_sigma_T,       qrf_sigma_P),
]:
    for tgt, score, resid in [('T_C', score_T, resid_T_arcpl),
                              ('P_kbar', score_P, resid_P_arcpl)]:
        c = _corr(score, np.abs(resid))
        rows.append({'method': method, 'target': tgt, **c})

ood_paradox_df = pd.DataFrame(rows)
ood_paradox_df.to_csv(RESULTS / 'nb10_ood_paradox_methods.csv', index=False)

# Persist per-sample OOD scores for all three methods for downstream use
pd.DataFrame({
    'index':          np.arange(len(df_arcpl)),
    'iso_anomaly':    iso_anomaly_arcpl,
    'mahal_pc5':      mahal,
    'qrf_sigma_T':    qrf_sigma_T,
    'qrf_sigma_P':    qrf_sigma_P,
    'abs_resid_T':    np.abs(resid_T_arcpl),
    'abs_resid_P':    np.abs(resid_P_arcpl),
}).to_csv(RESULTS / 'nb10_ood_scores_all_methods.csv', index=False)

print('OOD-method comparison on ArcPL (larger |Spearman r| = better error predictor):')
print(ood_paradox_df.round(4).to_string(index=False))

# Rank methods by mean |spearman_r| across both targets
rank = (ood_paradox_df.assign(abs_s=ood_paradox_df['spearman_r'].abs())
        .groupby('method')['abs_s'].mean().sort_values(ascending=False))
print('\nMean |Spearman r| across T and P (higher = better):')
print(rank.round(4).to_string())

print('\n' + '=' * 70)
print('OPERATOR DECISION REQUIRED: OOD method to canonize for NB08 filtering')
print('=' * 70)
print('  Option A: IsolationForest   (feature-space anomaly, current default)')
print('  Option B: Mahalanobis PC5   (linear manifold distance)')
print('  Option C: QRF sigma         (direct epistemic uncertainty)')
print('\nPick the method with the largest |Spearman r| vs |residual| - that is')
print('the one that actually predicts prediction error. See rank table above.')
print('=' * 70)

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Phase 9.EXT verification: confirm every absorbed-NB10 artifact was written.
_expected = [
    'nb10_two_pyroxene_benchmark.csv', 'nb10_h2o_dependence.csv',
    'nb10_h2o_engineered_arcpl.csv',   'nb10_h2o_engineered_test_rmse.csv',
    'nb10_iqr_uncertainty.csv',        'nb10_analytical_uncertainty.csv',
    'nb10_mc_per_sample.csv',          'nb10_ood_isoforest.csv',
    'nb10_arcpl_ood_scores.csv',       'nb10_ood_paradox_methods.csv',
    'nb10_ood_scores_all_methods.csv',
]
_missing = [f for f in _expected if not (RESULTS / f).exists()]
assert not _missing, f'Phase 9.EXT missing: {_missing}'
assert (MODELS / 'model_IsolationForest_opx_liq.joblib').exists()
assert (MODELS / 'model_RF_T_C_opx_liq_H2O.joblib').exists()
assert (MODELS / 'model_RF_P_kbar_opx_liq_H2O.joblib').exists()


In [ ]:
# Table 1: dataset summary (training core, opx-liq subset, full, ArcPL external)
df_core = pd.read_parquet(DATA_PROC / 'opx_clean_core.parquet')
df_full = pd.read_parquet(DATA_PROC / 'opx_clean_full.parquet')
df_liq_subset = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')

arcpl_path = RESULTS / 'nb04b_arcpl_predictions.csv'
df_arcpl = pd.read_csv(arcpl_path) if arcpl_path.exists() else pd.DataFrame()


def rng(s, fmt='{:.0f}-{:.0f}'):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if len(s) == 0:
        return 'NA'
    return fmt.format(s.min(), s.max())


table1_rows = [
    {'Dataset': 'ExPetDB core (training)', 'n_experiments': len(df_core),
     'n_studies': df_core['Citation'].nunique(),
     'T_range_C': rng(df_core['T_C']),
     'P_range_kbar': rng(df_core['P_kbar'], '{:.1f}-{:.1f}')},
    {'Dataset': 'ExPetDB opx-liq subset', 'n_experiments': len(df_liq_subset),
     'n_studies': df_liq_subset['Citation'].nunique(),
     'T_range_C': rng(df_liq_subset['T_C']),
     'P_range_kbar': rng(df_liq_subset['P_kbar'], '{:.1f}-{:.1f}')},
    {'Dataset': 'ExPetDB full (9 oxides)', 'n_experiments': len(df_full),
     'n_studies': df_full['Citation'].nunique(),
     'T_range_C': rng(df_full['T_C']),
     'P_range_kbar': rng(df_full['P_kbar'], '{:.1f}-{:.1f}')},
]
if len(df_arcpl) > 0 and 'T_C' in df_arcpl.columns:
    table1_rows.append({
        'Dataset': 'ArcPL external (LEPR _notinLEPR)',
        'n_experiments': len(df_arcpl),
        'n_studies': df_arcpl['Citation'].nunique() if 'Citation' in df_arcpl.columns else 'NA',
        'T_range_C': rng(df_arcpl['T_C']),
        'P_range_kbar': rng(df_arcpl.get('P_kbar', pd.Series(dtype=float)), '{:.1f}-{:.1f}'),
    })

table1 = pd.DataFrame(table1_rows)
table1.to_csv(RESULTS / 'table1_dataset_summary.csv', index=False)
print('Table 1: dataset summary')
print(table1.to_string(index=False))

In [ ]:
# Table 2: Phase 3R multi-seed model performance (mean +/- std over 20 seeds)
# Load the Phase 3R winners config (per_combo_winners). Previously this cell
# referenced config_3r without defining it, which failed under papermill.
config_3r = load_winning_config(RESULTS)
multi = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
multi_win = multi[multi['feature_set'] == WIN_FEAT].copy()

agg = (multi_win
       .groupby(['track', 'target', 'model_name'])
       .agg(rmse_test_mean=('rmse_test', 'mean'),
            rmse_test_std=('rmse_test', 'std'),
            mae_test_mean=('mae_test', 'mean'),
            r2_test_mean=('r2_test', 'mean'),
            n_seeds=('split_seed', 'nunique'))
       .reset_index())
agg['feature_set'] = WIN_FEAT

# Also include the per-combo winner tag from Phase 3R config for convenience.
per_combo = config_3r.get('per_combo_winners', {})
def _tag(row):
    key = f'{row.track}_{row.target}'
    w = per_combo.get(key, {})
    return 'winner' if w.get('model_name') == row.model_name else ''
agg['phase3r_tag'] = agg.apply(_tag, axis=1)

agg.to_csv(RESULTS / 'table2_model_performance.csv', index=False)
print('Table 2: Phase 3R multi-seed performance (winning feature set only)')
print(agg.round(3).to_string(index=False))

In [ ]:
# Table 3: Putirka (Thermobar) vs canonical ML
fair_path = RESULTS / 'nb04_putirka_comparison_fair.csv'
prac_path = RESULTS / 'nb04_putirka_comparison_practical.csv'
if fair_path.exists() and prac_path.exists():
    fair = pd.read_csv(fair_path); fair['Comparison'] = 'fair (same n)'
    prac = pd.read_csv(prac_path); prac['Comparison'] = 'practical (all test n)'
    table3 = pd.concat([fair, prac], ignore_index=True, sort=False)
    table3.to_csv(RESULTS / 'table3_putirka_vs_ml.csv', index=False)
    print('Table 3: Putirka vs ML')
    print(table3.round(3).to_string(index=False))
else:
    print('WARN: Putirka comparison CSVs missing; Table 3 skipped')

In [ ]:
# Table 4: validation strategies summary (Phase 5R pooled RMSE for canonical RF)
pooled = pd.read_csv(RESULTS / 'nb05_loso_pooled.csv')
per_fold = pd.read_csv(RESULTS / 'nb05_per_fold_rmse.csv')

# v8-fix: nb05 pool contains rows from multiple feature_sets; keep the
# full RF slice so downstream lookup sees whichever feature_set nb05
# actually wrote. (Historic bug: filtering to a fixed WIN_FEAT dropped
# every row when the canonical winner differed from the nb05 training
# feature set.)
pooled_rf = pooled[pooled['model_name'] == 'RF'].copy()
_pool_feats = sorted(pooled_rf['feature_set'].unique().tolist())
_pool_strats = sorted(pooled_rf['strategy'].unique().tolist())
print(f'v8-check: pooled_rf has {len(pooled_rf)} rows; '
      f'feature_sets={_pool_feats}; strategies={_pool_strats}')
per_fold_rf = per_fold[per_fold['model_name'] == 'RF'].copy()

per_fold_agg = (per_fold_rf
                .groupby(['strategy', 'target'])
                .agg(fold_median_rmse=('rmse', 'median'),
                     fold_mean_rmse=('rmse', 'mean'),
                     fold_q25=('rmse', lambda s: s.quantile(0.25)),
                     fold_q75=('rmse', lambda s: s.quantile(0.75)),
                     n_folds=('fold', 'nunique'))
                .reset_index())

table4 = pooled_rf.merge(per_fold_agg, on=['strategy', 'target'], how='left')
table4.to_csv(RESULTS / 'table4_validation_summary.csv', index=False)
print('Table 4: validation strategies (pooled + per-fold), canonical RF only')
print(table4.round(3).to_string(index=False))


In [ ]:
# Table 5: SHAP feature importance (top-10 by P importance).
# NB06 writes the canonical table5_shap_importance.csv; NB09 reads it here
# for manuscript compilation only. No rewrite.
table5 = pd.read_csv(RESULTS / 'table5_shap_importance.csv')
print('Table 5: top-10 SHAP features (read from NB06 output)')
print(table5.round(4).to_string(index=False))


In [ ]:
# Table 6: ArcPL external metrics
arcpl_metrics_path = RESULTS / 'nb04b_arcpl_metrics.csv'
if arcpl_metrics_path.exists():
    arcpl_metrics = pd.read_csv(arcpl_metrics_path)
    arcpl_metrics.to_csv(RESULTS / 'table6_arcpl_external.csv', index=False)
    print('Table 6: ArcPL external metrics')
    print(arcpl_metrics.round(3).to_string(index=False))
else:
    print('WARN: nb04b_arcpl_metrics.csv missing; Table 6 not built')

In [ ]:
# Table 7: Phase 7R bias correction A/B + QRF leakage A/B merged
ab_path = RESULTS / 'nb07_ab_test_report.csv'
qrf_path = RESULTS / 'nb07_qrf_ab_coverage.csv'
if ab_path.exists():
    ab = pd.read_csv(ab_path)
    ab.to_csv(RESULTS / 'table7_bias_correction.csv', index=False)
    print('Table 7: 4-track bias correction A/B')
    print(ab.round(3).to_string(index=False))
else:
    print('WARN: nb07_ab_test_report.csv missing; Table 7 not built')

if qrf_path.exists():
    qrf_ab = pd.read_csv(qrf_path)
    qrf_ab.to_csv(RESULTS / 'table7b_qrf_coverage.csv', index=False)
    print('Table 7b: QRF leakage A/B coverage')
    print(qrf_ab.round(3).to_string(index=False))
else:
    print('WARN: nb07_qrf_ab_coverage.csv missing; Table 7b not built')

In [ ]:
# NOTE: The nb10_* filename prefix below is retained for manuscript stability. These files are produced here by NB09 Phase 9.2-9.4 after the v6 archival of NB10.
# Tables 8-11: Phase 11 extended analyses
def _read_nonempty_csv(path):
    """Return DataFrame or None if file missing or has no parseable columns."""
    if not path.exists() or path.stat().st_size < 3:
        return None
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return None
    return df if len(df.columns) else None


t8 = _read_nonempty_csv(RESULTS / 'nb10_two_pyroxene_benchmark.csv')
if t8 is not None and len(t8):
    t8.to_csv(RESULTS / 'table8_two_pyroxene.csv', index=False)
    print('Table 8: two-pyroxene Thermobar benchmark')
    print(t8.round(3).to_string(index=False))
else:
    print('INFO: two-pyroxene benchmark empty (no Cpx in test set); Table 8 skipped')

t9 = _read_nonempty_csv(RESULTS / 'nb10_h2o_dependence.csv')
if t9 is not None and len(t9):
    t9.to_csv(RESULTS / 'table9_h2o_dependence.csv', index=False)
    print('Table 9: ArcPL H2O dependence')
    print(t9.round(4).to_string(index=False))
else:
    print('INFO: H2O dependence table empty; Table 9 skipped')

t10a = _read_nonempty_csv(RESULTS / 'nb10_iqr_uncertainty.csv')
t10b = _read_nonempty_csv(RESULTS / 'nb10_analytical_uncertainty.csv')
if t10a is not None and t10b is not None:
    t10a = t10a.assign(source='tree_IQR')
    t10b = t10b.assign(source='analytical_noise')
    t10 = pd.concat([t10a, t10b], ignore_index=True, sort=False)
    t10.to_csv(RESULTS / 'table10_uncertainty.csv', index=False)
    print('Table 10: per-sample uncertainty (tree IQR and analytical noise)')
    print(t10.round(3).to_string(index=False))
else:
    print('INFO: uncertainty tables missing; Table 10 skipped')

t11 = _read_nonempty_csv(RESULTS / 'nb10_ood_isoforest.csv')
if t11 is not None and len(t11):
    t11.to_csv(RESULTS / 'table11_ood_isoforest.csv', index=False)
    print('Table 11: ArcPL IsolationForest OOD filter')
    print(t11.round(3).to_string(index=False))
else:
    print('INFO: OOD table missing; Table 11 skipped')


In [ ]:
# Figure inventory (Phase F1 roster)
# v8-fix: canonical v7 figure roster (replaces legacy fig01-fig14 names)
CANONICAL_FIGURES = [
    {'num': 1,  'stem': 'fig_nb02_clusters',
     'caption': 'PCA decomposition of opx composition space, colored by k-means cluster.'},
    {'num': 2,  'stem': 'fig_nb03c_multiseed_rmse',
     'caption': 'Multi-seed test RMSE (mean +/- std over 20 seeds) per model x feature set.'},
    {'num': 3,  'stem': 'fig_nb04_method_benchmark_paired',
     'caption': 'Method benchmark on ArcPL three-phase paired scope. T RMSE (A), P RMSE (B, sorted independently), coverage (C).'},
    {'num': 4,  'stem': 'fig_nb04_diagnostic_encyclopedia_T',
     'caption': 'Per-method predicted-vs-observed T scatter, ArcPL three-phase paired scope.'},
    {'num': 5,  'stem': 'fig_nb04_diagnostic_encyclopedia_P',
     'caption': 'Per-method predicted-vs-observed P scatter, same scope.'},
    {'num': 6,  'stem': 'fig_nb04_arcpl_opx_liq_scatter',
     'caption': 'ArcPL external validation (n=204): forest and boosted families pred vs obs scatter + residuals.'},
    {'num': 7,  'stem': 'fig_nb04_h2o_sensitivity',
     'caption': 'Performance stratified by H2O reporting method (measured vs VBD).'},
    {'num': 8,  'stem': 'fig_nb04_opx_only_comparison',
     'caption': 'Eight-method opx-only comparison: 4 ML (ours, forest/boosted x opx-liq/opx-only) + 4 Putirka variants.'},
    {'num': 9,  'stem': 'fig_nb05_generalization',
     'caption': 'Generalization under three CV strategies: LOSO, Cluster-KFold, TargetBinKFold. Random-split reference lines dashed.'},
    {'num': 10, 'stem': 'fig_nb06_shap_P_beeswarm',
     'caption': 'SHAP attributions for P. Rows = features; dots = per-experiment contributions; color = feature value.'},
    {'num': 11, 'stem': 'fig_nb06_shap_T_beeswarm',
     'caption': 'Same as Fig 10 for T.'},
    {'num': 12, 'stem': 'fig_nb07_conformal_coverage',
     'caption': 'Conformal interval coverage at nominal 0.68 / 0.80 / 0.90 / 0.95 levels.'},
    {'num': 13, 'stem': 'fig_nb08_twopx_1to1',
     'caption': 'Cross-mineral validation: ML opx-only vs Jorgenson cpx-only on LEPR paired-pyroxene experiments.'},
]

expected_figures = [f"{spec['stem']}.png" for spec in CANONICAL_FIGURES]
existing = [f for f in expected_figures if (FIGURES / f).exists()]
missing = [f for f in expected_figures if not (FIGURES / f).exists()]
print(f'Found {len(existing)}/{len(expected_figures)} expected figures')
for m in missing:
    print(f'  MISSING: {m}')
inv_df = pd.DataFrame({
    'figure': expected_figures,
    'present': [f in existing for f in expected_figures],
})
inv_df.to_csv(RESULTS / 'figure_inventory.csv', index=False)


# Manuscript key results manifest
def _best_from_multi(track, target):
    sub = multi_win[(multi_win.track == track) & (multi_win.target == target)]
    if len(sub) == 0:
        return float('nan')
    return float(sub.groupby('model_name')['rmse_test'].mean().min())


def _pooled_rmse(strategy, target):
    sub = pooled_rf[(pooled_rf['strategy'] == strategy) &
                     (pooled_rf['target'] == target)]
    return float(sub['rmse'].iloc[0]) if len(sub) else float('nan')


manifest = {
    'winning_feature_set':        WIN_FEAT,
    'runner_up_feature_set':      config_3r.get('runner_up_feature_set', 'NA'),
    'best_opx_liq_T_RMSE_mean':   _best_from_multi('opx_liq',  'T_C'),
    'best_opx_liq_P_RMSE_mean':   _best_from_multi('opx_liq',  'P_kbar'),
    'best_opx_only_T_RMSE_mean':  _best_from_multi('opx_only', 'T_C'),
    'best_opx_only_P_RMSE_mean':  _best_from_multi('opx_only', 'P_kbar'),
    'loso_pooled_T_RMSE':         _pooled_rmse('LOSO',          'T_C'),
    'loso_pooled_P_RMSE':         _pooled_rmse('LOSO',          'P_kbar'),
    'cluster_pooled_T_RMSE':      _pooled_rmse('Cluster-KFold', 'T_C'),
    'cluster_pooled_P_RMSE':      _pooled_rmse('Cluster-KFold', 'P_kbar'),
    'gridded_pooled_T_RMSE':      _pooled_rmse('TargetBinKFold', 'T_C'),  # v8-fix: renamed from Gridded-PT
    'gridded_pooled_P_RMSE':      _pooled_rmse('TargetBinKFold', 'P_kbar'),  # v8-fix: renamed from Gridded-PT
}

if ab_path.exists():
    ab_df = pd.read_csv(ab_path)
    for tgt in ['T_C', 'P_kbar']:
        for track in ['1_raw_rf', '3_linear_ooffit', '4_piecewise_ooffit']:
            row = ab_df[(ab_df.target == tgt) & (ab_df.track == track)]
            if len(row):
                manifest[f'{track}_{tgt}_rmse_test'] = float(row['rmse_test'].iloc[0])

if qrf_path.exists():
    qdf = pd.read_csv(qrf_path)
    for tgt in ['T_C', 'P_kbar']:
        for tr in ['A_trainfit_leak', 'B_oof_honest', 'C_test']:
            row = qdf[(qdf.target == tgt) & (qdf.track == tr)]
            if len(row):
                manifest[f'qrf_{tr}_{tgt}_cov68'] = float(row['coverage_68'].iloc[0])

arcpl_metrics_path = RESULTS / 'nb04b_arcpl_metrics.csv'
if arcpl_metrics_path.exists():
    arc = pd.read_csv(arcpl_metrics_path)
    for tgt in ['T', 'P']:
        sub = arc[(arc['dataset'] == 'ArcPL all') & (arc['target'] == tgt)]
        if len(sub):
            manifest[f'arcpl_{tgt}_RMSE'] = float(sub['RMSE'].iloc[0])

pd.DataFrame([manifest]).to_csv(RESULTS / 'manuscript_key_results.csv', index=False)
print('KEY RESULTS:')
for k, v in manifest.items():
    print(f'  {k}: {v}')
print('\n=== NB09 complete ===')


## Inline manuscript summary

All tables and figures produced across the pipeline are rendered below in
a single scrollable view so the notebook itself is a standalone manuscript
artifact. Tables are printed as formatted DataFrames; figures are loaded
from `figures/` and displayed inline at native resolution.

In [ ]:
# Inline table summary: iterate every table*.csv produced and display.
from IPython.display import display, Markdown, Image

table_titles = {
    'table1_dataset_summary.csv':       'Table 1 - Dataset summary',
    'table2_model_performance.csv':     'Table 2 - Phase 3R multi-seed performance',
    'table3_putirka_vs_ml.csv':         'Table 3 - Putirka (Thermobar) vs ML',
    'table4_validation_summary.csv':    'Table 4 - Validation strategies (LOSO / Cluster-KFold / Gridded-PT)',
    'table5_shap_importance.csv':       'Table 5 - Top SHAP features',
    'table6_arcpl_external.csv':        'Table 6 - ArcPL external metrics',
    'table7_bias_correction.csv':       'Table 7 - Bias correction A/B',
    'table7b_qrf_coverage.csv':         'Table 7b - QRF coverage (A/B leakage check)',
    'table8_two_pyroxene.csv':          'Table 8 - Two-pyroxene Thermobar benchmark',
    'table9_h2o_dependence.csv':        'Table 9 - ArcPL H2O dependence',
    'table10_uncertainty.csv':          'Table 10 - Per-sample uncertainty (tree IQR + analytical noise)',
    'table11_ood_isoforest.csv':        'Table 11 - IsolationForest OOD filter',
    'manuscript_key_results.csv':       'Key-results manifest',
}

for fname, title in table_titles.items():
    path = RESULTS / fname
    if not path.exists() or path.stat().st_size < 3:
        display(Markdown(f'### {title}\n\n_(not produced this run)_'))
        continue
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        display(Markdown(f'### {title}\n\n_(file empty)_'))
        continue
    display(Markdown(f'### {title}'))
    display(df.round(4))


In [ ]:
# Inline figure gallery: display every figure present in figures/ in roster order,
# then any additional figures not on the canonical roster.
figure_captions = {
    'fig01_pt_distribution.png':         'Fig 1. P-T coverage of the ExPetDB training core.',
    'fig02_pca_biplot.png':              'Fig 2. PCA biplot of the opx composition space, colored by k-means cluster.',
    'fig03_pred_vs_obs_P.png':           'Fig 3. Canonical RF: predicted vs observed P on the held-out test set.',
    'fig04_pred_vs_obs_T.png':           'Fig 4. Canonical RF: predicted vs observed T on the held-out test set.',
    'fig05_model_comparison.png':        'Fig 5. Multi-seed test RMSE by model family and target (winning feature set).',
    'fig06_loso_pooled.png':             'Fig 6. Validation strategies: LOSO, Cluster-KFold, Gridded-PT pooled RMSE.',
    'fig07_shap_P_beeswarm.png':         'Fig 7. SHAP beeswarm for pressure predictions.',
    'fig08_shap_T_beeswarm.png':         'Fig 8. SHAP beeswarm for temperature predictions.',
    'fig09_shap_P_dependence.png':       'Fig 9a. SHAP dependence plots for top pressure features.',
    'fig09_shap_T_dependence.png':       'Fig 9b. SHAP dependence plots for top temperature features.',
    'fig10_putirka_vs_ml.png':           'Fig 10. Putirka (Thermobar) vs canonical ML: pred-vs-obs parity.',
    'fig11_bias_correction_residuals.png':'Fig 11. QRF bias-correction A/B residual distributions.',
    'fig12_natural_samples_geotherm.png':'Fig 12. Natural opx samples vs Hasterok and Chapman (2011) geotherms.',
    'fig13_ood_score_vs_residual.png':   'Fig 13. IsolationForest OOD score vs ArcPL residual.',
    'fig14_mc_vs_iqr_uncertainty.png':   'Fig 14. Analytical-noise sigma vs tree-IQR uncertainty.',
}

displayed = set()
for fname, caption in figure_captions.items():
    path = FIGURES / fname
    if path.exists():
        display(Markdown(f'**{caption}**'))
        display(Image(filename=str(path)))
        displayed.add(fname)
    else:
        display(Markdown(f'**{caption}** _(missing: {fname})_'))

# Catch any bonus figures not on the canonical roster.
extras = sorted(p.name for p in FIGURES.glob('*.png') if p.name not in displayed)
if extras:
    display(Markdown('### Additional figures'))
    for fname in extras:
        display(Markdown(f'**{fname}**'))
        display(Image(filename=str(FIGURES / fname)))


In [ ]:
# v8-fix: pipeline health check. Writes logs/pipeline_health.txt with
# PASS/FAIL per upstream notebook output. Intended to surface silent
# breakage before the manuscript is compiled.
from datetime import datetime as _dt
import json as _json

_health = []

def _check(name, ok, detail=''):
    status = 'PASS' if ok else 'FAIL'
    line = f'[{status}] {name}'
    if detail:
        line += f' -- {detail}'
    _health.append(line)
    return ok

# NB01
_check('NB01 cleaned core',
       (DATA_PROC / 'opx_clean_core.parquet').exists())
# NB02
_check('NB02 clusters fig', (FIGURES / 'fig_nb02_clusters.png').exists())
# NB03
_winners_path = RESULTS / 'nb03_per_family_winners.json'
_check('NB03 winners json', _winners_path.exists())
if _winners_path.exists():
    _w = _json.loads(_winners_path.read_text())
    _n = sum(len(_w.get(f, {})) for f in ['forest_family', 'boosted_family'])
    _check('NB03 winner count == 8', _n == 8, f'found {_n}')
# NB04 core outputs
for _f in ['nb04_method_benchmark_arcpl_paired.csv',
           'nb04_arcpl_opx_liq_metrics.csv',
           'nb04_arcpl_h2o_stratified_metrics.csv',
           'nb04_opx_only_comparison_all.csv']:
    _check(f'NB04 {_f}', (RESULTS / _f).exists())
for _f in ['fig_nb04_method_benchmark_paired.png',
           'fig_nb04_diagnostic_encyclopedia_T.png',
           'fig_nb04_diagnostic_encyclopedia_P.png',
           'fig_nb04_arcpl_opx_liq_scatter.png',
           'fig_nb04_h2o_sensitivity.png',
           'fig_nb04_opx_only_comparison.png']:
    _check(f'NB04 {_f}', (FIGURES / _f).exists())
# NB04 specific: Putirka opx in benchmark
_bench_path = RESULTS / 'nb04_method_benchmark_arcpl_paired.csv'
if _bench_path.exists():
    _b = pd.read_csv(_bench_path)
    _has_put_opx = any('Putirka 2008 opx' in m for m in _b['Method'])
    _check('NB04 benchmark includes Putirka opx methods', _has_put_opx,
           f'methods: {_b["Method"].tolist()}')
# NB05
_check('NB05 LOSO pooled', (RESULTS / 'nb05_loso_pooled.csv').exists())
_check('NB05 generalization fig',
       (FIGURES / 'fig_nb05_generalization.png').exists())
# NB06
_check('NB06 SHAP P', (FIGURES / 'fig_nb06_shap_P_beeswarm.png').exists())
_check('NB06 SHAP T', (FIGURES / 'fig_nb06_shap_T_beeswarm.png').exists())
# NB07
_check('NB07 bias correction null',
       (RESULTS / 'nb07_bias_correction_null_result.csv').exists() or
       (RESULTS / 'nb07_ab_test_main.csv').exists())
# NB08
_check('NB08 predictions',
       (RESULTS / 'nb08_natural_predictions.csv').exists() or
       (RESULTS / 'nb08_disagreement_metrics.csv').exists())
# NB09 manifest sanity
_mani_path = RESULTS / 'manuscript_key_results.csv'
if _mani_path.exists():
    _mani = pd.read_csv(_mani_path)
    _nan_cols = [c for c in _mani.columns if pd.isna(_mani[c].iloc[0])]
    _check('NB09 manifest has no NaN fields', len(_nan_cols) == 0,
           f'NaN fields: {_nan_cols}' if _nan_cols else 'all populated')
# NB09 OOD separation
_ood_path = RESULTS / 'nb10_ood_isoforest.csv'
if _ood_path.exists():
    _ood = pd.read_csv(_ood_path)
    _distinct = False
    for _t in ['T_C', 'P_kbar']:
        _sub = _ood[_ood['target'] == _t]
        if len(_sub) >= 2 and 'rmse' in _sub.columns:
            _vals = _sub['rmse'].round(3).unique()
            if len(_vals) > 1:
                _distinct = True
                break
    _check('NB09 OOD subsets have distinct RMSE', _distinct,
           'at least one target shows different all/in/out metrics')

_n_pass = sum(1 for x in _health if x.startswith('[PASS]'))
_n_fail = sum(1 for x in _health if x.startswith('[FAIL]'))
_summary = f'SUMMARY: {_n_pass} PASS, {_n_fail} FAIL of {len(_health)} checks'
print(chr(10).join(_health))
print(_summary)

_health_path = LOGS / 'pipeline_health.txt'
_health_path.parent.mkdir(parents=True, exist_ok=True)
_health_path.write_text(
    f'Pipeline health report -- {_dt.now().isoformat()}\n'
    + '=' * 60 + '\n'
    + chr(10).join(_health) + '\n'
    + _summary + '\n'
)
print(f'Wrote {_health_path}')
if _n_fail > 0:
    print(f'WARNING: {_n_fail} pipeline checks failed. See pipeline_health.txt.')
